# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

In [27]:
import pandas as pd
import control
import numpy as np
from sklearn.linear_model import LinearRegression

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [6]:
A = pd.read_csv('model-A.tsv', sep='\t')
A

,index,x,v,a
0,x,1,1,0
1,v,0,1,1
2,a,0,0,1


In [7]:
A = A.set_index('index').to_numpy()
A

array([[1, 1, 0],
       [0, 1, 1],
       [0, 0, 1]])

In [8]:
B = pd.read_csv('model-B.tsv', sep='\t')
B

,index,u
0,x,0
1,v,0
2,a,1


In [9]:
B = B.set_index('index').to_numpy()
B

array([[0],
       [0],
       [1]])

In [10]:
Q = pd.read_csv('cost-Q.tsv', sep='\t')
Q

,index,x,v,a
0,x,5,0,0
1,v,0,1,0
2,a,0,0,2


In [11]:
Q = Q.set_index('index').to_numpy()
Q

array([[5, 0, 0],
       [0, 1, 0],
       [0, 0, 2]])

In [12]:
R = pd.read_csv('cost-R.tsv', sep='\t')
R

,index,u
0,u,5


In [13]:
R = R.set_index('index').to_numpy()
R

array([[5]])

In [14]:
K, S, E= control.dlqr(A, B, Q, R)
K

array([[0.33445985, 1.30445413, 1.85813088]])

In [16]:
K_df = pd.DataFrame(K, columns=['x', 'v', 'a'])
K_df

,x,v,a
0,0.33446,1.304454,1.858131


Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [17]:
K_df.to_csv('control-K-intended.tsv', index=False)

Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [20]:
qc_log = pd.read_csv('qc-log.tsv', sep='\t')
qc_log.drop(columns='index', inplace=True)
qc_log

,x,v,a,u
0,-5.000000e+00,0.000000e+00,0.000000e+00,1.702188e+00
1,-5.000000e+00,-1.702188e-02,1.531969e+00,-1.263200e+00
2,-5.018724e+00,1.452683e+00,5.482855e-01,-1.249321e+00
3,-3.420773e+00,1.840779e+00,-5.212749e-01,-2.121274e-01
4,-1.395916e+00,1.163611e+00,-7.643170e-01,4.528954e-01
...,...,...,...,...
95,-1.855920e-18,1.097460e-18,-1.592936e-19,-4.843645e-19
96,-6.487133e-19,8.412290e-19,-6.111510e-19,3.189634e-19
97,2.766386e-19,1.733230e-19,-3.851990e-19,4.316642e-19
98,4.672940e-19,-2.142650e-19,-3.522116e-20,1.881712e-19


In [22]:
state_cols = ['x', 'v', 'a']
states = qc_log[state_cols].to_numpy()
states[:5]

array([[-5.        ,  0.        ,  0.        ],
       [-5.        , -0.01702188,  1.53196898],
       [-5.01872407,  1.45268284,  0.54828547],
       [-3.42077294,  1.84077896, -0.52127491],
       [-1.39591608,  1.16361118, -0.76431704]])

In [24]:
u_check = (-states @ K.T).ravel() # convert 2d arr with nested array of len 1 into a 1d arr
u_check[:5]

array([ 1.67229926, -1.15209535, -1.23518258, -0.2885035 ,  0.36920157])

In [25]:
qc_check_df = pd.DataFrame({
    'index': qc_log.index,
    'u_check': u_check
})
qc_check_df

,index,u_check
0,0,1.672299e+00
1,1,-1.152095e+00
2,2,-1.235183e+00
3,3,-2.885035e-01
4,4,3.692016e-01
...,...,...
95,95,-5.148677e-19
96,96,2.552224e-19
97,97,3.971337e-19
98,98,1.886533e-19


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [26]:
qc_check_df.to_csv('qc-check.tsv', index=False)

Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [ ]:
log_actions = qc_log['u'].to_numpy()

model = LinearRegression(fit_intercept=False)
model.fit(states, log_actions) # predict actions using the other 3 to estimate coefs

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",False
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](3,)","[-0.34,-1.3 ,-1.95]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float,0
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,3
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(3)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](3,)","[9.61,2.51,1.05]"


In [38]:
K_actual = -model.coef_.reshape(1, -1) # convert to 2d arr
K_actual

array([[0.34043755, 1.30012023, 1.95011696]])

In [39]:
K_actual_df = pd.DataFrame(K_actual, columns=state_cols)
K_actual_df

,x,v,a
0,0.340438,1.30012,1.950117


Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [40]:
K_actual_df.to_csv('control-K-actual.tsv', index=False)

Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [43]:
qc_train = pd.read_csv('qc-train.tsv', sep='\t')
qc_train.drop(columns='index', inplace=True)
qc_train

,x,v,a,u
0,-5.000000,0.000000,0.000000,1.729856
1,-5.000000,-0.017299,1.556871,-1.311911
2,-5.019028,1.476577,0.531837,-1.198850
3,-3.394793,1.846154,-0.493944,-0.297565
4,-1.364024,1.195267,-0.811147,0.472619
...,...,...,...,...
95,0.089539,-0.162778,0.122030,-0.008091
96,-0.089517,-0.030491,0.126952,-0.174120
97,-0.123056,0.094904,-0.017061,0.038104
98,-0.018662,0.068824,0.015527,-0.062828


In [63]:
# We need to predict future state based on a current state. So we need to offset the X and Y
X = qc_train[:-1].to_numpy()
X[-5:]

array([[ 0.22299847, -0.12132642, -0.05427907,  0.2019304 ],
       [ 0.08953942, -0.1627782 ,  0.12203038, -0.00809074],
       [-0.0895166 , -0.03049061,  0.12695175, -0.17412003],
       [-0.12305627,  0.09490382, -0.0170611 ,  0.03810446],
       [-0.01866207,  0.06882434,  0.0155268 , -0.06282786]])

In [62]:
# We need to predict future state based on a current state. So we need to offset the X and Y
y = qc_train[state_cols][1:].to_numpy()
y[:5]

array([[-5.        , -0.01729856,  1.55687058],
       [-5.01902842,  1.47657746,  0.53183744],
       [-3.39479321,  1.84615378, -0.49394382],
       [-1.36402405,  1.19526743, -0.81114707],
       [-0.04922988,  0.30042479, -0.46690501]])

In [64]:
offset_model = LinearRegression(fit_intercept=False)
offset_model.fit(X, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",False
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](3, 4)","[[ 1. , 1.1 , 0. , 0. ], [ 0. , 0.9 , 0.95,-0.01], [-0. ,-0. , 1.1 , 0.9 ]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float,0
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,4
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(4)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](4,)","[9.74,3.83,2.69,0.37]"


In [65]:
A_new = offset_model.coef_[:, :3]
A_new

array([[ 1.00000000e+00,  1.10000000e+00,  4.27435864e-15],
       [ 9.49905150e-18,  9.00000000e-01,  9.50000000e-01],
       [-5.53830553e-16, -1.52655666e-15,  1.10000000e+00]])

In [66]:
B_new = offset_model.coef_[:, 3:]
B_new

array([[ 6.10622664e-16],
       [-1.00000000e-02],
       [ 9.00000000e-01]])

In [73]:
A_new_df = pd.DataFrame(A_new, index=state_cols, columns=state_cols)
A_new_df.index.name = 'index'
A_new_df

,x,v,a
index,,,
x,1.000000e+00,1.100000e+00,4.274359e-15
v,9.499052e-18,9.000000e-01,9.500000e-01
a,-5.538306e-16,-1.526557e-15,1.100000e+00


In [74]:
B_new_df = pd.DataFrame(B_new, index=state_cols, columns=['u'])
B_new_df.index.name = 'index'
B_new_df

,u
index,
x,6.106227e-16
v,-1.000000e-02
a,9.000000e-01


Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [75]:
A_new_df.to_csv('model-A-new.tsv', sep='\t')

In [76]:
B_new_df.to_csv('model-B-new.tsv', sep='\t')

Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.